# VLDB First Upgrade: Dynamic Screening
CPU-only Stage 2 campaign. This notebook does not rerun the completed static opportunity sweep.

In [ ]:
from pathlib import Path
import shutil
import zipfile

input_root = Path('/kaggle/input')
root = Path('/kaggle/working/SIGMOD')
if root.exists():
    shutil.rmtree(root)
root.mkdir(parents=True)
bundles = list(input_root.rglob('kaggle_upload_dynamic_screening.zip'))
if len(bundles) == 1:
    with zipfile.ZipFile(bundles[0]) as archive:
        archive.extractall(root)
elif len(bundles) == 0:
    configs = list(input_root.rglob('experiments/configs/dynamic/screening.json'))
    assert len(configs) == 1, 'Expected one unpacked dynamic-screening source tree'
    source_root = configs[0].parents[3]
    for item in source_root.iterdir():
        target = root / item.name
        if item.is_dir():
            shutil.copytree(item, target)
        else:
            shutil.copy2(item, target)
else:
    raise AssertionError(f'Expected at most one dynamic-screening ZIP, found {len(bundles)}')
print(root)
print((root / 'experiments/configs/dynamic/screening.json').read_text())

In [ ]:
!uname -a
!lscpu
!free -h
!g++ --version

In [ ]:
!bash {root}/scripts/linux/build_release.sh --execute
!{root}/build/prepared/test_v2

In [ ]:
import subprocess
smoke = root / 'results/raw/kaggle/dynamic-screening-smoke.csv'
smoke.parent.mkdir(parents=True, exist_ok=True)
subprocess.run([str(root / 'build/prepared/bench_unified'), '--run-id', 'dynamic-screening-smoke', '--output', str(smoke), '--variant', 'STATIC-REGIONAL', '--family', 'shifting', '--records', '1000', '--operations', '10000', '--phase-length', '1000', '--hot-fraction', '0.05', '--zipf', '0.99', '--fanout', '64', '--seed', '11'], check=True)
print(smoke.read_text())

In [ ]:
# 1,620 measured runs: 3 phase lengths x 3 hotspot sizes x 3 skews x 5 seeds x 2 variants x 6 fanouts.
!bash {root}/scripts/linux/run_campaign.sh experiments/configs/dynamic/screening.json --execute kaggle

In [ ]:
!python3 {root}/analysis/phase_oracle.py --raw {root}/results/raw/kaggle --output {root}/results/processed/dynamic_screening_phase_oracle.csv
!python3 {root}/experiments/select_regimes.py --family dynamic_screening --raw {root}/results/raw/kaggle --output {root}/results/manifests/dynamic_screening_selected_regimes.json
!python3 {root}/analysis/aggregate.py --raw {root}/results/raw/kaggle --output {root}/results/processed/dynamic_screening_summary.csv

In [ ]:
import json
manifest = root / 'results/manifests/dynamic_screening_selected_regimes.json'
print(json.dumps(json.loads(manifest.read_text()), indent=2))
print('main CSVs:', len([p for p in (root / 'results/raw/kaggle').rglob('*.csv') if not p.name.endswith('.phases.csv')]))
print('phase CSVs:', len(list((root / 'results/raw/kaggle').rglob('*.phases.csv'))))

In [ ]:
results_zip = shutil.make_archive('/kaggle/working/results', 'zip', root / 'results')
resume_root = Path('/kaggle/working/resume_bundle')
if resume_root.exists():
    shutil.rmtree(resume_root)
shutil.copytree(root / 'results', resume_root / 'results')
shutil.copy2(root / 'experiments/configs/dynamic/screening.json', resume_root / 'screening.json')
resume_zip = shutil.make_archive('/kaggle/working/resume', 'zip', resume_root)
print(results_zip)
print(resume_zip)